AHL-D CRNN checkpoint evaluated on Fidel handwritten, no retraining.

In [ ]:
#@title Test AHL-D 29K CRNN on Fidel Handwritten Dataset

!pip install -q datasets jiwer

import os
import zipfile
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from jiwer import cer, wer

from google.colab import drive

drive.mount("/content/drive")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/"
    "amharic_htr_research/ocr/research/crnn/"
    "ahld29k/best_checkpoint.pt"
)

OUTPUT_PATH = (
    "/content/drive/MyDrive/"
    "amharic_htr_research/ocr/research/crnn/"
    "ahld29k_fidel_predictions.csv"
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

state_dict = checkpoint["model_state_dict"]
char_to_idx = checkpoint["char_to_idx"]
idx_to_char = {int(k): v for k, v in checkpoint["idx_to_char"].items()}
blank_idx = checkpoint["blank_idx"]

class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.MaxPool2d((2, 1), (2, 1)),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.MaxPool2d((2, 1), (2, 1))
        )

        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            dropout=0.2
        )

        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = F.adaptive_avg_pool2d(x, (1, x.size(3)))
        x = x.squeeze(2)
        x = x.permute(2, 0, 1)
        x, _ = self.rnn(x)
        return self.classifier(x)

model = CRNN(num_classes=len(char_to_idx) + 1).to(DEVICE)
model.load_state_dict(state_dict)
model.eval()

print(
    f"Model loaded | best epoch {checkpoint.get('epoch')} | "
    f"AHL-D CER {checkpoint.get('best_cer'):.4f}"
)

FIDEL_BASE_URL = (
    "https://huggingface.co/datasets/upanzi/fidel-dataset/resolve/main/"
)

FIDEL_DIR = "/content/fidel_data"
TRAIN_ZIP = os.path.join(FIDEL_DIR, "train-clean.zip")
TRAIN_LABELS = os.path.join(FIDEL_DIR, "train_labels.csv")
TRAIN_IMAGES_DIR = os.path.join(FIDEL_DIR, "train-clean")

os.makedirs(FIDEL_DIR, exist_ok=True)

if not os.path.exists(TRAIN_ZIP):
    print("Downloading train-clean.zip...")
    !wget -q --show-progress -O "{TRAIN_ZIP}" "{FIDEL_BASE_URL}train-clean.zip"

if not os.path.exists(TRAIN_LABELS):
    print("Downloading train_labels.csv...")
    !wget -q --show-progress -O "{TRAIN_LABELS}" "{FIDEL_BASE_URL}train_labels.csv"

labels = pd.read_csv(TRAIN_LABELS)

print(f"Original labels: {len(labels)}")
print(f"Types:\n{labels['type'].value_counts()}")

labels = labels[
    labels["type"].astype(str).str.lower() == "handwritten"
].reset_index(drop=True)

print(f"Handwritten labels: {len(labels)}")

if len(labels) == 0:
    raise RuntimeError("No handwritten samples were found in train_labels.csv.")

if not os.path.exists(TRAIN_IMAGES_DIR):
    print("Extracting train-clean.zip...")
    with zipfile.ZipFile(TRAIN_ZIP, "r") as zip_ref:
        zip_ref.extractall(FIDEL_DIR)

image_extensions = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff"
}

image_paths = []

for root, dirs, files in os.walk(FIDEL_DIR):
    for filename in files:
        extension = os.path.splitext(filename)[1].lower()

        if extension in image_extensions:
            image_paths.append(os.path.join(root, filename))

print(f"Found {len(image_paths)} image files.")

if len(image_paths) == 0:
    raise RuntimeError(
        "No image files were found after extracting train-clean.zip."
    )

image_column = "image_filename"
text_column = "line_text"

if image_column not in labels.columns:
    raise RuntimeError(
        f"Missing image column '{image_column}'. "
        f"Available columns: {labels.columns.tolist()}"
    )

if text_column not in labels.columns:
    raise RuntimeError(
        f"Missing text column '{text_column}'. "
        f"Available columns: {labels.columns.tolist()}"
    )

image_lookup = {}

for path in image_paths:
    filename = os.path.basename(path)
    stem = os.path.splitext(filename)[0]

    image_lookup[filename] = path
    image_lookup[stem] = path

matched_rows = []
unmatched_rows = []

for _, row in labels.iterrows():

    image_identifier = str(row[image_column])

    candidates = [
        image_identifier,
        os.path.basename(image_identifier),
        os.path.splitext(os.path.basename(image_identifier))[0]
    ]

    image_path = None

    for candidate in candidates:
        if candidate in image_lookup:
            image_path = image_lookup[candidate]
            break

    if image_path is not None:
        matched_rows.append({
            "image_path": image_path,
            "text": str(row[text_column])
        })
    else:
        unmatched_rows.append(image_identifier)

print(f"Matched handwritten samples: {len(matched_rows)}")
print(f"Unmatched handwritten labels: {len(unmatched_rows)}")

if len(matched_rows) == 0:
    raise RuntimeError(
        "No handwritten image/label pairs could be matched."
    )

if len(unmatched_rows) > 0:
    print("First unmatched image identifiers:")

    for identifier in unmatched_rows[:10]:
        print(identifier)

class FidelDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        image = Image.open(sample["image_path"])
        text = str(sample["text"])

        image = image.convert("L")
        image = image.resize(
            (512, 64),
            Image.Resampling.LANCZOS
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        image = torch.from_numpy(image).unsqueeze(0)

        return image, text

def collate_fn(batch):
    images = torch.stack([x[0] for x in batch])
    texts = [x[1] for x in batch]

    return images, texts

loader = DataLoader(
    FidelDataset(matched_rows),
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn
)

def greedy_decode(outputs):
    predictions = outputs.argmax(dim=2)

    results = []

    for sequence in predictions.permute(1, 0):

        text = []
        previous = blank_idx

        for index in sequence.tolist():

            if index != blank_idx and index != previous:
                text.append(idx_to_char.get(index, ""))

            previous = index

        results.append("".join(text))

    return results

references = []
predictions = []

print("Running inference...")

with torch.no_grad():

    for images, texts in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        outputs = model(images)

        batch_predictions = greedy_decode(outputs)

        references.extend(texts)
        predictions.extend(batch_predictions)

fidel_cer = cer(
    references,
    predictions
)

fidel_wer = wer(
    references,
    predictions
)

results = pd.DataFrame({
    "ground_truth": references,
    "prediction": predictions
})

results.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print(
    f"Fidel handwritten | "
    f"Samples {len(results)} | "
    f"CER {fidel_cer:.4f} | "
    f"WER {fidel_wer:.4f}"
)

for i in range(min(10, len(results))):

    print(f"GT: {references[i]}")
    print(f"PR: {predictions[i]}")

print(f"Saved: {OUTPUT_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model loaded | best epoch 20 | AHL-D CER 0.0462
/content/fidel_data 100%[===================>]  10.56G   101MB/s    in 2m 43s  
Original labels: 112716
Types:
type
synthetic      55478
handwritten    34820
typed          22418
Name: count, dtype: int64
Handwritten labels: 34820
Extracting train-clean.zip...
Found 112716 image files.
Matched handwritten samples: 34820
Unmatched handwritten labels: 0
Running inference...
Fidel handwritten | Samples 34820 | CER 0.4938 | WER 0.8571
GT: ለምሳሌ  ያህል  ፣ ልጆቻችሁ  በትምህርት  ቤት ጥሩ ውጤት  ቢያመጡ  የጉልበተኞች  የጥቃት  ዒላማ  

PR: ለመሳል ጭሠስ፣ በጀታችው በትጨርነቅ ቤት ጠቡ ውጠት በመጡ የትታ በተሞች ውቃት ከተማ
GT: ቸው  ቀኑን  በለውጥ  ስለ መበየን  በጽ / ቤታቸው የቀረበውን  አጀንዳ  በተመለከተ  ምልአተ  ጉባኤው  

PR: ቸው ቀገ በሰውጥ ስለ መጠጣ በቁመሙታቸው ንቁረበውን እጅን በተመለሰተ መብከተ ጉባው
GT: የተለያዩ  አማራጮችን  አይቷል።  አንዲት  ወላድ  ሴት ( ከአራስ  ቤት ከወጣች  5 ወር ቢሆናት  ነው ) 

PR: የተለኑ ከመራመፍ። አይቷል" አርቱት ወብድ ስት ፎጥሰርጉ ቤት ከውጣቁ ናውር በ